# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oduntanfolake/FlyRank-ML-first-assignment-solution-/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
from google.colab import userdata
from huggingface_hub import HfApi

token = userdata.get("HF_TOKEN")

print("Token loaded:", token is not None)
print("Token length:", len(token) if token else 0)

api = HfApi(token=token)
me = api.whoami()

print("Hugging Face login:", me["name"])

Token loaded: True
Token length: 37
Hugging Face login: Fbeva


In [3]:
import duckdb

con = duckdb.connect()

con.execute(
    "CREATE SECRET (TYPE huggingface, TOKEN ?)",
    [token]
)

print("DuckDB → Hugging Face connection ready.")

DuckDB → Hugging Face connection ready.


In [4]:
import duckdb

con = duckdb.connect()

con.execute(
    "CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN ?)",
    [token]
)

print("DuckDB connected to Hugging Face.")

DuckDB connected to Hugging Face.


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Feature vector: The feature frame uses three consistently observed GSC performance signals: gsc_impressions, gsc_clicks, and gsc_sum_position. These were selected because they have no missing values in the March 2026 development slice. No categorical fields are required for this feature vector.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the March 2026 feature vector

features = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_sum_position
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
""").df()

print("Feature frame shape:", features.shape)
features.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (9841378, 6)


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_sum_position
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,67
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,616
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,28
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,25


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Feature notes

gsc_impressions — Number of Google Search Console impressions observed for the content. It has no missing values in the March 2026 slice, so no fill is required. It is available at the decision moment because it represents observed search performance for the selected reporting period.

gsc_clicks — Number of clicks received from Google Search. It has no missing values in the March 2026 slice, so no fill is required. It is available at the decision moment because it represents observed clicks for the selected reporting period.

gsc_sum_position — Aggregate Google Search Console position signal for the content. It has no missing values in the March 2026 slice, so no fill is required. It is available at the decision moment because it represents observed search-position information for the selected reporting period.


Categorical handling: No categorical features are included in this feature vector, so no categorical encoding is required.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check feature meaning and missingness in the March 2026 feature frame

print("Feature columns:")
print(["gsc_impressions", "gsc_clicks", "gsc_sum_position"])

print("\nMissing values:")
print(features[[
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position"
]].isna().sum())

print("\nFeature frame shape:", features.shape)

Feature columns:
['gsc_impressions', 'gsc_clicks', 'gsc_sum_position']

Missing values:
gsc_impressions     0
gsc_clicks          0
gsc_sum_position    0
dtype: int64

Feature frame shape: (9841378, 6)


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Leakage test: I deliberately created a label-derived feature from the engagement-opportunity proxy to demonstrate leakage. This feature would not be available independently at the decision moment because it contains information derived from the target being ranked. I compare the deliberately leaky feature with the honest feature set, then remove the leaky feature and keep only information that would be available at decision time.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Deliberate leakage experiment

import numpy as np

leak_test = features.copy()

leak_test["engagement_opportunity_proxy"] = (
    (1 - (leak_test["gsc_clicks"] / leak_test["gsc_impressions"].replace(0, np.nan)))
    + (leak_test["gsc_sum_position"] / 100)
).fillna(0)

leak_test["leaking_feature"] = leak_test["engagement_opportunity_proxy"]

print("Leakage test columns:")
print([
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "leaking_feature"
])

print("\nProxy/leaking-feature comparison:")
print(
    leak_test[
        ["engagement_opportunity_proxy", "leaking_feature"]
    ].head()
)

print(
    "\nMaximum difference between proxy and leaking feature:",
    np.abs(
        leak_test["engagement_opportunity_proxy"]
        - leak_test["leaking_feature"]
    ).max()
)

Leakage test columns:
['gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'leaking_feature']

Proxy/leaking-feature comparison:
   engagement_opportunity_proxy  leaking_feature
0                         1.670            1.670
1                         1.000            1.000
2                         7.152            7.152
3                         1.280            1.280
4                         1.250            1.250

Maximum difference between proxy and leaking feature: 0.0


##*Leakage finding:*

The deliberately leaky feature reproduced the engagement-opportunity proxy exactly, with zero difference. This demonstrates why a label-derived feature cannot be used as an honest model input: it contains the information being ranked. The leaky feature and proxy were removed, leaving only the three observed GSC features that are available at the decision moment.

In [ ]:
leaking_error = np.abs(
    leak_test["engagement_opportunity_proxy"]
    - leak_test["leaking_feature"]
)

print("Maximum error using leaky feature:", leaking_error.max())
print("Mean error using leaky feature:", leaking_error.mean())

# Remove the deliberately leaky feature
honest_features = leak_test.drop(
    columns=["engagement_opportunity_proxy", "leaking_feature"]
)

print("\nHonest feature columns after removing leakage:")
print(honest_features.columns.tolist())

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*
Excluded fields

client_hash_id and content_hash_id excluded: because they are identifiers, not meaningful performance signals, and using them as model features could cause memorization rather than useful learning.
report_date and month - excluded as direct model features because they describe the observation period rather than the content's engagement performance.

GA4 fields such as ga4_pageviews, ga4_sessions, ga4_users, and ga4_engaged_sessions excluded: because GA4 availability is uneven in the March slice, so using these fields would reduce the consistency of the feature set.

Other session-source fields such as sessions_organic, sessions_direct, sessions_referral, sessions_social, sessions_paid, and sessions_ai excluded: because they are not part of the selected core GSC feature set for this first feature vector.

AI source fields such as ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, and ai_other excluded: because they are not required for the current engagement-opportunity feature vector.

gsc_avg_position excluded: because it has substantial missingness in the March slice, making it less consistently available than gsc_sum_position.

scroll_events excluded: from the first feature vector because it also has substantial missingness and is not consistently observed across the slice.

The deliberately created leaky_feature excluded: because it directly contains the engagement-opportunity proxy and therefore leaks the information being ranked.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Verify that excluded/leaky fields are not part of the honest feature set

# Verify that excluded/leaky fields are not part of the honest feature set

excluded_fields = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "month",
    "gsc_avg_position",
    "scroll_events",
    "leaking_feature",
    "engagement_opportunity_proxy"
]

honest_feature_fields = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position"
]

print("Honest model features:")
print(honest_feature_fields)

print("\nExcluded fields checked:")
print(excluded_fields)

print("\nLeakage removed:")
print("leaking_feature" not in honest_feature_fields)
print("engagement_opportunity_proxy" not in honest_feature_fields)

Honest model features:
['gsc_impressions', 'gsc_clicks', 'gsc_sum_position']

Excluded fields checked:
['client_hash_id', 'content_hash_id', 'report_date', 'month', 'gsc_avg_position', 'scroll_events', 'leaking_feature', 'engagement_opportunity_proxy']

Leakage removed:
True
True


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.